# Meeting 12: rows and columns

**CS 301, Introduction to Data Science, NJIT.** Unit II, meeting 12, the second notebook.

A matrix looks like a table, but the memory of a computer is one long line of numbers. This notebook finds where each number of a matrix is kept in that line, and measures what it costs to read the same matrix row by row and column by column: first with loops in Python, then with NumPy. The same numbers, read in another order, can take many times as long, and so can a matrix one column smaller than its neighbour.

Do it **before meeting 12**, after the first notebook of the meeting, and upload it with the first one.

## How this works

- **This notebook needs no GPU.** Leave its runtime as it is.
- **First** choose **Save a copy in Drive** in the **File** menu, so that your changes are kept, and run the cells in order, from the top. To run a cell, click it and press **Shift + Enter**.
- **You may work with your group** of meeting 11. **Every student uploads their own copy.**
- **Six parts**, with code to run. Four places ask you to write a function or to have an AI assistant write one, marked <font color="#c00000"><b>Your turn</b></font>, each with a **check** whose message says what to look at.
- **Five cells ask for your own words**: one marked <font color="#c00000"><b>Your prompt</b></font>, the prompt you give an AI assistant, and four marked <font color="#c00000"><b>Your thinking</b></font>. **They are what this notebook is graded on**, with the six cells of the first notebook. Write them yourself, even when your group worked together.
- **Nothing is posted from this notebook.** At the end, upload it to Canvas together with the first notebook, as the last section says.

<font color="#c00000" size="5"><b>Your turn.</b></font> Double-click this cell and replace this line with your name and the names of the other members of your group.

In [ ]:
# Setup. Run this cell first.
import ast
import glob
import inspect
import textwrap
import time

import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(0)           # a source of random numbers


def best_time(f, repeats=3):
    """The shortest of a few runs of f(), in seconds: the run least disturbed by anything else on the machine."""
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        f()
        times.append(time.perf_counter() - start)
    return min(times)


def caches():
    """The caches of this processor, as (name, MB), where the system tells them; an empty list where it does not."""
    found = []
    for folder in sorted(glob.glob("/sys/devices/system/cpu/cpu0/cache/index*")):
        try:
            kind = open(folder + "/type").read().strip()
            level = open(folder + "/level").read().strip()
            size = open(folder + "/size").read().strip()
        except OSError:
            continue
        if kind != "Instruction":
            factor = {"K": 1024, "M": 1024 ** 2}.get(size[-1], 1)
            found.append((f"L{level}", float(size.rstrip("KM")) * factor / 1e6))
    return found


def has_loop(function):
    """True when the text of the function contains a loop."""
    try:
        tree = ast.parse(textwrap.dedent(inspect.getsource(function)))
    except (OSError, TypeError, SyntaxError):
        return False
    loops = (ast.For, ast.While, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    return any(isinstance(node, loops) for node in ast.walk(tree))


def reads(function):
    """What the code of a function does, from its syntax (comments and docstrings do not count): "column" when it takes
    a column such as X[:, j], "transpose" when it uses .T, "sum" when it calls a sum function such as X.sum or np.sum."""
    try:
        tree = ast.parse(textwrap.dedent(inspect.getsource(function)))
    except (OSError, TypeError, SyntaxError):
        return set()
    found = set()
    for node in ast.walk(tree):
        if (isinstance(node, ast.Subscript) and isinstance(node.slice, ast.Tuple) and node.slice.elts
                and isinstance(node.slice.elts[0], ast.Slice)):
            found.add("column")
        if isinstance(node, ast.Attribute) and node.attr == "T":
            found.add("transpose")
        if isinstance(node, ast.Call):
            name = node.func.attr if isinstance(node.func, ast.Attribute) else getattr(node.func, "id", "")
            if name in ("sum", "reduce", "einsum", "add_reduce"):
                found.add("sum")
    return found


print("Setup is done. Caches of this processor:",
      ", ".join(f"{name} {mb:g} MB" for name, mb in caches()) or "the system of this machine does not tell them.")

## <font color="#1967d2"><b>Part 1: A matrix is one line of numbers</b></font>

Every byte of the memory of a computer has an **address**, a number, and the addresses run in one sequence. A NumPy array keeps its numbers in one block of consecutive addresses, one number after another, and remembers how to read the block as a table. The next cell makes a matrix with 3 rows and 4 columns and shows the line in which NumPy stores it.

In [ ]:
X = np.arange(12.0).reshape(3, 4)        # the numbers 0 to 11, as 3 rows of 4
print(X)
print("the line in memory:", X.ravel())
print("bytes for one number:", X.itemsize, "  strides:", X.strides)

**How to read the result.** NumPy stores the rows one after another: first row 0, then row 1, then row 2. This is called **row order**. The **strides** say how many bytes to step in the line to move by one position along each dimension: 32 bytes to go down one row, past the 4 numbers of 8 bytes of a row, and 8 bytes to go right by one column.

So the number in row i and column j of a matrix with m columns is at position i × m + j of the line: i whole rows come before it, and then j numbers of its own row.

<font color="#c00000" size="5"><b>Your turn.</b></font> Complete the function `position` in the next cell: it returns the position in the line of the number in row `i` and column `j` of a matrix with `m` columns, stored in row order. Then run the check.

In [ ]:
def position(i, j, m):
    """The position, in the line of memory, of the number in row i and column j of a matrix with m columns."""
    return ...     # Your turn: replace the three dots

In [ ]:
Y = np.arange(35.0).reshape(5, 7)        # 5 rows of 7: a matrix that is not square
assert position(0, 0, 7) is not ..., "Replace the three dots in the function position, and run that cell again."
line = Y.ravel()
wrong = [(i, j) for i in range(5) for j in range(7) if position(i, j, 7) != i * 7 + j]
if wrong and all(position(i, j, 7) == j * 5 + i for i, j in wrong):
    raise AssertionError("That is the position in a line that stores the columns one after another. Here the rows are "
                         "stored one after another: before row i come i whole rows of m numbers each.")
assert not wrong, (f"position{wrong[0] + (7,)} returns {position(*wrong[0], 7)}, but that number is at position "
                   f"{int(Y[wrong[0]])} of the line. Before row i come i whole rows of m numbers; then j more.")
print("Correct: every number of a matrix with 5 rows and 7 columns is where position says.")

**The transpose is the same line, read in another way.** `X.T` makes no copy and moves no number: it swaps the two strides, so that one step along a row of `X.T` is a step down a column of `X`. Other languages store matrices the other way, **column order**, one column after another; NumPy can do that too, with `np.asfortranarray`. The next cell shows both.

In [ ]:
XT = X.T
print("X.T has shape", XT.shape, "and strides", XT.strides, "  same memory as X:", np.shares_memory(X, XT))
C = np.asfortranarray(X)                 # the same table, stored one column after another
print("in column order, the line is:", C.ravel(order="K"), "  strides:", C.strides)

## <font color="#1967d2"><b>Part 2: Loops in Python, by row and by column</b></font>

The two functions in the next cell add up every number of a matrix. The first goes along the rows, as the line is stored. The second goes down the columns: each step jumps a whole row ahead in the line. The cell times both on a matrix of 2,000 by 2,000 numbers, 32 MB.

In [ ]:
def sum_by_rows(X):
    """Every number of X, row after row: the order of the line."""
    n, m = X.shape
    total = 0.0
    for i in range(n):
        for j in range(m):
            total += X[i, j]
    return total


def sum_by_columns(X):
    """Every number of X, column after column: a jump of a whole row at each step."""
    n, m = X.shape
    total = 0.0
    for j in range(m):
        for i in range(n):
            total += X[i, j]
    return total


n = 2000
X = rng.random((n, n))
t_rows = best_time(lambda: sum_by_rows(X), repeats=2)
t_columns = best_time(lambda: sum_by_columns(X), repeats=2)
print(f"{n} x {n} numbers, {X.nbytes / 1e6:.0f} MB")
print(f"by rows:    {t_rows:.3f} s, {t_rows / n ** 2 * 1e9:.0f} nanoseconds for each number")
print(f"by columns: {t_columns:.3f} s, {t_columns / n ** 2 * 1e9:.0f} nanoseconds for each number")
print(f"by columns / by rows: {t_columns / t_rows:.2f}")

**The same time in both orders.** Answer in the next cell, in a few sentences: each pass of these loops takes tens of nanoseconds. Where does that time go? Why can the order in which the numbers are visited not show in it?

<font color="#c00000" size="5"><b>Your thinking.</b></font> Double-click this cell and replace this line with your answer, in your own words.

## <font color="#1967d2"><b>Part 3: Rows and columns, with NumPy inside the loop</b></font>

The loop over the numbers was the expensive part, so this part leaves one loop in Python and gives the rest to NumPy. `sum_by_rows_numpy` loops over the rows; each row, `X[i, :]`, is a block of m numbers side by side in the line, and NumPy adds them up.

<font color="#c00000" size="5"><b>Your turn.</b></font> Complete `sum_by_columns_numpy` in the next cell, the same function with a loop over the **columns**: each column of `X` is `X[:, j]`, the m numbers at the same position of every row. Then run the check.

In [ ]:
def sum_by_rows_numpy(X):
    """Every number of X: a loop over the rows, and NumPy adds up each row."""
    total = 0.0
    for i in range(X.shape[0]):
        total += X[i, :].sum()
    return total


def sum_by_columns_numpy(X):
    """Every number of X: a loop over the columns, and NumPy adds up each column."""
    total = 0.0
    ...            # Your turn: replace the three dots with a loop over the columns of X
    return total

In [ ]:
T = rng.random((300, 200))               # 300 rows and 200 columns: a matrix that is not square
assert has_loop(sum_by_columns_numpy), "sum_by_columns_numpy needs a loop over the columns: for j in range(X.shape[1]): ..."
assert reads(sum_by_columns_numpy) & {"column", "transpose"}, (
    "Inside the loop, add up one column at a time: X[:, j].sum(), the numbers at position j of every row.")
try:
    total = sum_by_columns_numpy(T)
except IndexError:
    raise AssertionError("The loop asks for a column that X does not have. A matrix with n rows and m columns has m "
                         "columns: range(X.shape[1]).") from None
assert np.isclose(total, T.sum()), (
    "The total is not the sum of every number. A matrix with n rows and m columns has m columns: range(X.shape[1]).")
print("Correct: sum_by_columns_numpy adds up every number, one column at a time.")

The next cell times the two functions on square matrices of five sizes, from 256 by 256 numbers (half a megabyte) to 4,096 by 4,096 (134 MB), and draws the time of each number in both orders. Where this notebook can read the sizes of the caches of its processor, dotted lines mark them. It takes up to half a minute.

In [ ]:
sizes = [256, 512, 1024, 2048, 4096]
by_rows, by_columns, megabytes = [], [], []
for n in sizes:
    X = rng.random((n, n))
    megabytes.append(X.nbytes / 1e6)
    by_rows.append(best_time(lambda: sum_by_rows_numpy(X)) / n ** 2 * 1e9)
    by_columns.append(best_time(lambda: sum_by_columns_numpy(X)) / n ** 2 * 1e9)
del X

print(f"{'size':>6s}{'MB':>8s}{'by rows, ns a number':>23s}{'by columns':>13s}{'columns / rows':>17s}")
for n, mb, r, c in zip(sizes, megabytes, by_rows, by_columns):
    print(f"{n:6d}{mb:8.1f}{r:23.2f}{c:13.2f}{c / r:17.1f}")

fig, ax = plt.subplots(figsize=(9, 4.6))
ax.plot(megabytes, by_rows, marker="o", color="#2a78d6", label="by rows")
ax.plot(megabytes, by_columns, marker="s", linestyle="--", color="#eb6834", label="by columns")
for name, mb in caches():
    ax.axvline(mb, color="#888888", linestyle=":", linewidth=1)
    ax.annotate(name, (mb, max(by_columns)), xytext=(4, 0), textcoords="offset points", fontsize=9, va="top")
ax.set_xscale("log")
ax.set_xlabel("MB of the matrix (each mark on the axis is 10 times the one before)")
ax.set_ylabel("nanoseconds for each number")
ax.set_title("The same sum, read by rows and by columns")
ax.grid(True, which="major", color="#e1e0d9", linewidth=0.8)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

**How to read the result.** The memory does not hand the processor one number at a time. It hands the cache a **cache line**: 64 bytes, 8 numbers of this matrix that sit side by side in the line. Reading a row uses all 8 numbers of every line that it fetches, and when the reading goes in order, the processor fetches the next lines before they are asked for. Reading a column uses 1 number of each line and jumps a whole row ahead for the next. While the matrix fits in the cache, the other 7 numbers of each line are still there when the next column needs them. When it does not fit, they are gone, and every number of a column costs a trip to the main memory.

**Where the order begins to matter.** Answer in the next cell, in a few sentences: at which size did reading by columns become much slower than reading by rows? Compare it with the sizes of the caches: the dotted lines, or Part 5 of the first notebook. Explain, with cache lines, why a column costs more than a row once the matrix no longer fits in the cache.

<font color="#c00000" size="5"><b>Your thinking.</b></font> Double-click this cell and replace this line with your answer, in your own words.

## <font color="#1967d2"><b>Part 4: A power of two, and one more</b></font>

Sizes that are powers of two, 1,024, 2,048, 4,096, are everywhere: images, layers of networks and batches are often made that size. The next cell reads matrices by columns, with your `sum_by_columns_numpy` of Part 3, at three such sizes and at their two neighbours, one number smaller and one number larger. It takes up to half a minute.

Before you run it, predict: a matrix with 2,049 columns has more numbers than a matrix with 2,048. Should it take longer for **each** number?

In [ ]:
groups = [1024, 2048, 4096]
sizes_pow2 = [n for k in groups for n in (k - 1, k, k + 1)]
pow2_ns = {n: float("inf") for n in sizes_pow2}
for k in groups:
    matrices = {n: rng.random((n, n)) for n in (k - 1, k, k + 1)}
    for _ in range(5):                   # five rounds over the three sizes: the best time of each is kept
        for n, X in matrices.items():
            start = time.perf_counter()
            sum_by_columns_numpy(X)
            pow2_ns[n] = min(pow2_ns[n], (time.perf_counter() - start) / n ** 2 * 1e9)
del matrices, X

print(f"{'size':>6s}{'by columns, ns a number':>26s}")
for n in sizes_pow2:
    print(f"{n:6d}{pow2_ns[n]:26.2f}" + ("   <- a power of two" if n in groups else ""))

fig, ax = plt.subplots(figsize=(9, 4.4))
for g, k in enumerate(groups):
    for d, n in enumerate((k - 1, k, k + 1)):
        x = 4 * g + d
        ax.bar(x, pow2_ns[n], color="#eb6834" if n == k else "#2a78d6", width=0.8)
        ax.annotate(f"{n}", (x, pow2_ns[n]), xytext=(0, 4), textcoords="offset points", ha="center", fontsize=9)
ax.set_xticks([4 * g + 1 for g in range(len(groups))])
ax.set_xticklabels([f"around {k:,}" for k in groups])
ax.set_ylabel("nanoseconds for each number, by columns")
ax.set_title("Reading by columns: a power of two (orange) and its neighbours (blue)")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

**How to read the result.** A cache does not let a line go anywhere it likes. It is divided into **sets**: the address of a line decides which set it goes to, and a set holds only a few lines, 8 to 16 on the processors of today. Lines that follow each other in memory go to sets that follow each other, so a block read in order spreads over all the sets, and reading by rows never runs into this.

A column is not read in order: its numbers are a whole row apart. When a row is 2,048 numbers long, 16,384 bytes, a power of two, the numbers of a column fall into the same few sets. Only a dozen or so of their lines can stay; the others push them out, and the neighbours that the next column needs are gone, although the whole matrix would fit in the cache. With 2,049 numbers in a row, each row moves the column by 8 more bytes, its lines spread over the sets, and they stay.

**The fix is one more number in each row.** Libraries that compute with matrices often store a matrix of 2,048 columns in rows of 2,049 numbers, and leave the last number of each row unused. This is called **padding**. In NumPy, `np.zeros((n, m + 1))[:, :m]` is such a matrix: its shape is (n, m), and its rows are m + 1 numbers apart in the line.

<font color="#c00000" size="5"><b>Your turn.</b></font> Complete the function `padded` in the next cell: replace the three dots with an array of zeros with n rows and m + 1 columns. The function then keeps its first m columns and copies the numbers of `X` into them. Run the check, which also times the columns of a matrix of 2,048 by 2,048 numbers, stored as it is and padded.

In [ ]:
def padded(X):
    """A copy of X stored in rows one number longer than X's: the same table, with one unused number after each row."""
    n, m = X.shape
    P = ...              # Your turn: an array of zeros with n rows and m + 1 columns
    P = P[:, :m]         # its first m columns: shape (n, m), the rows m + 1 numbers apart
    P[:] = X             # the numbers of X
    return P

In [ ]:
T = rng.random((300, 200))
try:
    P = padded(T)
except TypeError:
    raise AssertionError("Replace the three dots in the function padded with np.zeros((n, m + 1)), and run that cell again.") from None
assert P.shape == T.shape, (
    f"padded returned shape {P.shape}; it must keep the shape of X, {T.shape}. Add the extra number to each row: "
    "np.zeros((n, m + 1)), not one more row.")
assert np.array_equal(P, T), "padded must hold the numbers of X."
assert P.strides[0] == (T.shape[1] + 1) * T.itemsize, (
    f"The rows of this copy are {P.strides[0] // T.itemsize} numbers apart in the line; they must be m + 1 = {T.shape[1] + 1}: "
    "make the zeros with m + 1 columns.")
print("Correct: padded keeps the table and stores each row one number longer.")

n = 2048
X = rng.random((n, n))
XP = padded(X)
t_plain = t_padded = float("inf")
for _ in range(5):
    start = time.perf_counter()
    sum_by_columns_numpy(X)
    t_plain = min(t_plain, time.perf_counter() - start)
    start = time.perf_counter()
    sum_by_columns_numpy(XP)
    t_padded = min(t_padded, time.perf_counter() - start)
print(f"\nThe columns of a matrix of {n:,} by {n:,} numbers:")
print(f"stored as it is, rows of {n:,} numbers:  {t_plain * 1e3:7.1f} milliseconds")
print(f"padded, rows of {n + 1:,} numbers:         {t_padded * 1e3:7.1f} milliseconds")
print(f"as it is / padded: {t_plain / t_padded:.1f}")

**One more number.** Answer in the next cell, in a few sentences, with the words *cache line* and *set*: why was reading by columns slower at a power of two than at its neighbours, and why does one more number in each row make it fast again? Would you expect the same effect when the matrix is read by rows? Why, or why not?

<font color="#c00000" size="5"><b>Your thinking.</b></font> Double-click this cell and replace this line with your answer, in your own words.

## <font color="#1967d2"><b>Part 5: Column sums, read by rows</b></font>

Often the result itself is by columns: the mean of every column of a table of data, for example, needs the sum of each column. The obvious loop goes down each column, which Part 3 found slow. But the same sums can be collected while reading the matrix **by rows**: keep m running sums, one for each column, and add each row to them.

<font color="#c00000" size="5"><b>Your turn.</b></font> Ask an AI assistant to write the function `column_sums_by_rows`. State all of these requirements in your prompt:

- the function is named `column_sums_by_rows` and takes one argument, `X`, a NumPy array with n rows and m columns;
- it returns a NumPy array of m numbers: the sum of each column of `X`;
- it starts from an array of m zeros and has one loop, over the **rows** of `X`, which adds each whole row to that array;
- it has no loop over the columns, does not take a column `X[:, j]`, and does not call `X.sum` or `np.sum`.

Paste your prompt into the next cell, and the function into the code cell after it. Then run the check.

<font color="#c00000" size="5"><b>Your prompt.</b></font> Double-click this cell and replace this line with the prompt you gave to the AI assistant. If the first function did not pass the check, add one sentence on what you changed.

In [ ]:
# Your turn: paste the function column_sums_by_rows here, then run this cell.

In [ ]:
assert "column_sums_by_rows" in globals(), "Paste the function column_sums_by_rows into the cell above and run that cell."
assert has_loop(column_sums_by_rows), "column_sums_by_rows needs one loop, over the rows of X."
assert not reads(column_sums_by_rows) & {"column", "transpose"}, (
    "column_sums_by_rows must read X by rows only: no column X[:, j] and no transpose.")
assert "sum" not in reads(column_sums_by_rows), (
    "column_sums_by_rows must add the rows itself, in its loop: no X.sum, np.sum or other sum function.")
T = rng.random((300, 200))               # 300 rows and 200 columns
result = np.asarray(column_sums_by_rows(T))
assert result.shape == (200,), (
    f"column_sums_by_rows returned an array of shape {result.shape}. It must hold one sum for each column: (m,), here (200,).")
assert np.allclose(result, T.sum(axis=0)), "These are not the sums of the columns. Add each whole row to the running sums."
print("Correct: column_sums_by_rows returns the sum of every column, reading the matrix by rows.")

n = 4096
X = rng.random((n, n))
t_columns = best_time(lambda: [X[:, j].sum() for j in range(n)])
t_rows = best_time(lambda: column_sums_by_rows(X))
t_numpy = best_time(lambda: X.sum(axis=0))
print(f"\nThe sums of the {n:,} columns of a matrix of {X.nbytes / 1e6:.0f} MB:")
print(f"a loop over the columns, X[:, j].sum():   {t_columns * 1e3:8.1f} milliseconds")
print(f"column_sums_by_rows, a loop over the rows: {t_rows * 1e3:8.1f} milliseconds")
print(f"NumPy's own X.sum(axis=0):                 {t_numpy * 1e3:8.1f} milliseconds")

## <font color="#1967d2"><b>Part 6: What NumPy does by itself</b></font>

`X.sum(axis=0)` sums down the columns, and it was as fast as the loop over the rows, or faster: NumPy reads the matrix in the order of its line whatever the result asks for. Not every operation can do that. The next cell times four pairs of operations on a matrix of 4,096 by 4,096 numbers: in each pair, the second reads the same numbers in another order.

In [ ]:
n = 4096
X = rng.random((n, n))
F = np.asfortranarray(X)                 # the same matrix, stored in column order
v = rng.random(n)
pairs = [("X.sum(axis=1)", lambda: X.sum(axis=1), "X.sum(axis=0)", lambda: X.sum(axis=0)),
         ("X @ v", lambda: X @ v, "F @ v, F in column order", lambda: F @ v),
         ("X.copy()", lambda: X.copy(), "X.T.copy()", lambda: X.T.copy()),
         ("X + X", lambda: X + X, "X + X.T", lambda: X + X.T)]
print(f"{'operation':16s}{'ms':>8s}    {'the same numbers, another order':32s}{'ms':>8s}{'ratio':>8s}")
for name_a, a, name_b, b in pairs:
    ta, tb = best_time(a), best_time(b)
    print(f"{name_a:16s}{ta * 1e3:8.1f}    {name_b:32s}{tb * 1e3:8.1f}{tb / ta:8.1f}")

**When NumPy cannot choose the order.** Answer in the next cell, in a few sentences: why does `X.sum(axis=0)`, a sum down the columns, cost no more than `X.sum(axis=1)`, while your loop over the columns in Part 3 cost so much more? Why does `X + X.T` cost several times as much as `X + X`, although both add the same number of pairs?

<font color="#c00000" size="5"><b>Your thinking.</b></font> Double-click this cell and replace this line with your answer, in your own words.

**Every array of this unit is a line.** An image of h rows and w columns of pixels, with three colours, has shape (h, w, 3) and is stored pixel after pixel: the red, green and blue numbers of one pixel sit side by side, and the red channel `image[:, :, 0]` is every third number of the line. The layers of PyTorch that read images expect a batch with the colours first, (N, 3, h, w), so that each colour of an image is one block. A shape says how to read the line; the order of the line decides which readings are fast.

## Upload your notebooks

**Every student uploads their own copies of both notebooks of meeting 12, before the meeting begins.** They are graded on the cells marked <font color="#c00000"><b>Your prompt</b></font> and <font color="#c00000"><b>Your thinking</b></font>: six in the first notebook and five in this one. Check that each holds your own words and not the line that was there.

1. In each notebook, open the **File** menu, choose **Download**, and then **Download .ipynb**.
2. On Canvas, open the module of meeting 12, choose **Upload Your Class Work - Meeting 12**, and upload **both files** in the same submission.